In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [2]:

# Load dataset in chunks
dataset_path = "./datasets/complaints.csv"  # Update with actual dataset path
chunk_size = 512  # Adjust based on memory availability

# Initialize TF-IDF Vectorizer (Fitted only once)
tfidf = TfidfVectorizer(stop_words="english", max_features=5000, ngram_range=(1,2))

# Initialize model for online learning
model = SGDClassifier(loss="hinge", random_state=42)  # Linear SVM-like classifier
first_batch = True  # Track first batch
batch_accuracies = []  # Store batch accuracy
all_classes = set()  # To collect all unique product classes
dict_y_to_replace = {
                 'Credit reporting or other personal consumer reports':'Credit reporting',
                 'Credit reporting, credit repair services, or other personal consumer reports':'Credit reporting',
                  'Credit card': 'Credit card or prepaid card',
                  'Prepaid card': 'Credit card or prepaid card',
                  'Debt or credit management': 'Debt collection,credit management',
                  'Debt collection': 'Debt collection,credit management',
                  'Bank account': 'Bank account,checking and saving account or service',
                  'Money transfer': 'Money transfer, virtual currency, or money service',
                  'Student loan': 'Consumer loan',
                  'Payday loan, title loan, or personal loan': 'Consumer loan',
                  'Payday loan, title loan, personal loan, or advance loan': 'Consumer loan',
                  'Mortgage': 'Consumer loan',
                  'Vehicle loan': 'Consumer loan',
                  'Vehicle loan or lease': 'Consumer loan',
                  'Other financial service': 'Other financial service',
                  'Virtual currency': 'Money transfer, virtual currency, or money service'
                  }

# Read dataset in chunks
reader = pd.read_csv(dataset_path, chunksize=chunk_size)


In [18]:
chunk = next(df)  # Get first chunk from TextFileReader
print(chunk['Product'].unique())


['Credit reporting or other personal consumer reports' 'Debt collection'
 'Money transfer, virtual currency, or money service'
 'Checking or savings account'
 'Payday loan, title loan, personal loan, or advance loan' 'Credit card'
 'Mortgage' 'Vehicle loan or lease' 'Debt or credit management']


In [ ]:

all_classes = set()  # Initialize as set
for chunk in pd.read_csv(dataset_path, chunksize=chunk_size):
    if "Product" in chunk and "Consumer complaint narrative" in chunk:
        all_classes.update(set(chunk["Product"].dropna().unique()))  # Update set with unique labels

# Convert to sorted list for consistent encoding 
# print(all_classes)
all_classes = sorted(list(all_classes))
print(len(all_classes))
all_classes_df = pd.DataFrame({"Product": all_classes})
all_classes_df["Product"] = all_classes_df["Product"].replace(dict_y_to_replace)
all_classes = all_classes_df.Product.unique()
print(f"Total unique classes found: {len(all_classes)}")
joblib.dump(all_classes, "models/unique_predictable_class.pkl")


In [ ]:
a = ["asua","roger","asua_roger","kamanzi"]
a_df = pd.DataFrame({"Product":a})
print(a)
a_df['Product'] = a_df['Product'].replace({"asua":"asua_roger","roger":"asua_roger"})
print(a_df['Product'].unique())

In [ ]:
# **First Pass: Fit TF-IDF Once & Get All Classes**
# Read dataset in chunks again for training
print("Training model in batches...")
for chunk in reader:
    # Filter and clean data
    df1 = chunk[['Product', 'Consumer complaint narrative']].dropna()
    df1.columns = ['Product', 'Consumer_complaint']
    df1['Product'] = df1['Product'].replace(dict_y_to_replace)
    # If chunk is too small, skip it
    if df1.shape[0] < 2:  # Minimum 2 samples needed for splitting
        print("Skipping small chunk...")
        continue

    # Encode labels using previously collected classes
    df1["Product"] = df1["Product"].astype("category").cat.set_categories(all_classes)
    y_encoded = df1["Product"].cat.codes  # Convert categories to numerical labels

    # Split into train/test
    X_train, X_test, y_train, y_test = train_test_split(
        df1["Consumer_complaint"], y_encoded, test_size=0.25, random_state=42
    )

    # **TF-IDF Transformation**
    if first_batch:
        X_train_tfidf = tfidf.fit_transform(X_train)  # Fit TF-IDF only once
        first_batch = False
    else:
        X_train_tfidf = tfidf.transform(X_train)  # Use transform after first batch

    X_test_tfidf = tfidf.transform(X_test)  # Always transform test data
    print(f"AllClasses::{len(all_classes)}")
    # **Train incrementally**
    model.partial_fit(X_train_tfidf, y_train, classes=np.arange(len(all_classes)))

    # Predict batch
    y_pred = model.predict(X_test_tfidf)

    # Calculate batch accuracy
    batch_acc = accuracy_score(y_test, y_pred)
    batch_accuracies.append(batch_acc)
    print(f"Batch Accuracy: {batch_acc:.4f}")

In [24]:
# Print average batch accuracy
print(f"\nAverage Batch Accuracy: {np.mean(batch_accuracies):.4f}")

# **Save Model & TF-IDF Vectorizer**
joblib.dump(model, "models/sgd_model.pkl")
joblib.dump(tfidf, "models/tfidf_vectorizer.pkl")
print("\nModel and vectorizer saved successfully!")


Average Batch Accuracy: 0.8484

Model and vectorizer saved successfully!


In [23]:
print(all_classes)

['Bank account or service' 'Checking or savings account' 'Consumer Loan'
 'Credit card or prepaid card' 'Credit reporting'
 'Debt collection,credit management'
 'Money transfer, virtual currency, or money service' 'Money transfers'
 'Consumer loan' 'Other financial service' 'Payday loan']


In [6]:

# Load saved model and vectorizer
model_saved = joblib.load("models/sgd_model.pkl")
tfidf_saved = joblib.load("models/tfidf_vectorizer.pkl")
all_classes_saved = joblib.load("models/unique_predictable_class.pkl")

In [9]:

# Example new complaints for prediction
new_complaints = [
    """
    I am writing to borrow money for my study
    """
]

# Convert text to TF-IDF features (must match the training process)
new_complaints_tfidf = tfidf_saved.transform(new_complaints)

# Predict categories
predicted_labels = model_saved.predict(new_complaints_tfidf)
print(all_classes_saved)

# Print results
for complaint, label in zip(new_complaints, predicted_labels):
    print(label, ":", len(all_classes_saved))
    print(f"Predicted Category: {all_classes_saved[label]}\n")

['Bank account or service' 'Checking or savings account' 'Consumer Loan'
 'Credit card or prepaid card' 'Credit reporting'
 'Debt collection,credit management'
 'Money transfer, virtual currency, or money service' 'Money transfers'
 'Consumer loan' 'Other financial service' 'Payday loan']
7 : 11
Predicted Category: Money transfers

